In [1]:
import pandas as pd
import geopandas as gpd
import json
import os
import numpy as np

In [4]:
gcs_path = "https://storage.googleapis.com/wetlands-gap-map/original_data"
analysis_data_folder = 'updated_typology_csv'

In [5]:
locations = gpd.read_file(f'{gcs_path}/locations_simplified.geojson')
locations_sahel = locations[locations['type'] == 'global']
locations_countries = locations[locations['type'] == 'admin_region']
locations_wpda = locations[locations['type'] == 'wdpa']
locations_basins = locations[locations['type'] == 'hydro_basin']
print(f'Total locations: {len(locations)}')
print(f'Total Sahel locations: {len(locations_sahel)}')
print(f'Total country locations: {len(locations_countries)}')
print(f'Total WDPA locations: {len(locations_wpda)}')
print(f'Total basin locations: {len(locations_basins)}')

Total locations: 2068
Total Sahel locations: 1
Total country locations: 26
Total WDPA locations: 2024
Total basin locations: 17


In [34]:
# Column names dictionary
cols_dict = {
    '11': 'permopen',
    '12': 'seasopen',
    '13': 'ephopen',
    '14': 'saline',
    '17': 'lake',
    '18': 'seas_lake',
    '19': 'eph_lake',
    '26': 'perinuveg',
    '27': 'seainuveg',
    '28': 'ephinuveg',
    '32': 'agri',
    '34': 'reservoir',
    '41': 'mangrove',
    '42': 'mudflat',
    '128': 'shallow',
    '43': 'other'
}

wetland_types = list(cols_dict.values())
wetland_types.remove('other')

## Wetland type per country

In [39]:
file_path = f'{gcs_path}/{analysis_data_folder}/admin_region_areas.csv'
get_wetlands = pd.read_csv(file_path)
#replace column names with the dictionary
get_wetlands = get_wetlands.rename(columns=cols_dict)

#group categories
get_wetlands['seas_lake'] = get_wetlands['seas_lake'] + get_wetlands['15']
get_wetlands['saline'] = get_wetlands['saline'] + get_wetlands['16']
get_wetlands['agri'] = get_wetlands['agri'] + get_wetlands['31'] + get_wetlands['33']
get_wetlands['reservoir'] = get_wetlands['reservoir'] + get_wetlands['35'] + get_wetlands['36']

get_wetlands = get_wetlands.drop(columns=['15', '16', '31', '33', '35', '36', 'other'])

get_wetlands['total'] = get_wetlands[wetland_types].sum(axis=1)

#clean names
get_wetlands['Country'] = get_wetlands['Country'].replace({'The Gambia': 'Gambia'})
#Right join to preserve countries only in the locations_countries dataframe
get_wetlands = get_wetlands.merge(locations_countries[['name', 'code']], left_on='Country', right_on='name', how='right')
get_wetlands = get_wetlands.drop(columns=['name'])
get_wetlands['location_id'] = get_wetlands['code'].apply(lambda x: 'adminregion_' + x.lower())

get_wetlands.head()

,Country,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,shallow,total,code,location_id
0,Senegal,204910.29,143463.69,42353.64,332.55,26354.52,4522.77,3328.92,11460.33,108794.97,50312.79,69208.83,5989.86,172101.24,21246.93,206759.79,1071141.12,SEN,adminregion_sen
1,Gambia,48123.18,12514.23,415.71,0.00,219.69,21.33,47.52,2547.72,50198.67,15355.44,2607.75,0.27,78100.29,7791.66,124499.16,342442.62,GMB,adminregion_gmb
2,Kenya,41599.26,262270.26,78160.32,754345.71,394836.30,4796.46,9500.94,6014.52,121057.47,164236.86,103789.62,15802.47,54379.71,30144.96,127666.26,2168601.12,KEN,adminregion_ken
3,South Sudan,48208.23,492621.12,119927.07,134.19,71593.74,11616.66,6103.17,178421.67,7008218.37,5646269.70,60432.03,17975.34,0.00,0.00,0.00,13661521.29,SSD,adminregion_ssd
4,Mauritania,10907.82,103820.22,46998.72,1100.70,573.12,3114.27,393.12,224.73,42477.30,23834.07,32530.86,18068.94,310.41,4885.74,19967.04,309207.06,MRT,adminregion_mrt


In [40]:
#calculate percantage of each wetland type per country
for wetland in wetland_types:
    get_wetlands[wetland + '_perc'] = round(get_wetlands[wetland] / get_wetlands['total'] * 100, 2)

get_wetlands.drop(columns= wetland_types + ['total'], inplace=True)
get_wetlands.columns = get_wetlands.columns.str.replace('_perc', '')
get_wetlands.drop(columns=['code', 'Country'], inplace=True)
get_wetlands.head()

,location_id,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,shallow
0,adminregion_sen,19.13,13.39,3.95,0.03,2.46,0.42,0.31,1.07,10.16,4.70,6.46,0.56,16.07,1.98,19.30
1,adminregion_gmb,14.05,3.65,0.12,0.00,0.06,0.01,0.01,0.74,14.66,4.48,0.76,0.00,22.81,2.28,36.36
2,adminregion_ken,1.92,12.09,3.60,34.78,18.21,0.22,0.44,0.28,5.58,7.57,4.79,0.73,2.51,1.39,5.89
3,adminregion_ssd,0.35,3.61,0.88,0.00,0.52,0.09,0.04,1.31,51.30,41.33,0.44,0.13,0.00,0.00,0.00
4,adminregion_mrt,3.53,33.58,15.20,0.36,0.19,1.01,0.13,0.07,13.74,7.71,10.52,5.84,0.10,1.58,6.46


In [41]:
get_wetlands_long = get_wetlands.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_wetlands_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_wetlands_long.head(3)

,location_id,wetland_type,percentage
274,adminregion_ben,agri,11.95
170,adminregion_ben,eph_lake,0.08
248,adminregion_ben,ephinuveg,38.36


In [42]:
get_wetlands_long['wetland_type'].unique()

array(['agri', 'eph_lake', 'ephinuveg', 'ephopen', 'lake', 'mangrove',
       'mudflat', 'perinuveg', 'permopen', 'reservoir', 'saline',
       'seainuveg', 'seas_lake', 'seasopen', 'shallow'], dtype=object)

## Wetland types by wdpa

In [50]:
file_path = f'{gcs_path}/{analysis_data_folder}/wdpa_areas.csv'
get_wdpa = pd.read_csv(file_path)
#replace column names with the dictionary
get_wdpa = get_wdpa.rename(columns=cols_dict)

#group categories
get_wdpa['seas_lake'] = get_wdpa['seas_lake'] + get_wdpa['15']
get_wdpa['saline'] = get_wdpa['saline'] + get_wdpa['16']
get_wdpa['agri'] = get_wdpa['agri'] + get_wdpa['31'] + get_wdpa['33']
get_wdpa['reservoir'] = get_wdpa['reservoir'] + get_wdpa['35'] + get_wdpa['36']

get_wdpa = get_wdpa.drop(columns=['15', '16', '31', '33', '35', '36', 'other'])

get_wdpa['total'] = get_wdpa[wetland_types].sum(axis=1)

get_wdpa = get_wdpa.merge(locations_wpda[['name', 'id']], left_on='Country', right_on='name', how='right')
get_wdpa = get_wdpa.drop(columns=['name'])
get_wdpa.rename(columns={'id': 'location_id'}, inplace=True)

#calculate percantage of each wetland type per country
for wetland in wetland_types:
    get_wdpa[wetland + '_perc'] = round(get_wdpa[wetland] / get_wdpa['total'] * 100, 2)

get_wdpa.drop(columns= wetland_types + ['total'], inplace=True)
get_wdpa.columns = get_wdpa.columns.str.replace('_perc', '')
get_wdpa.drop(columns=['Country'], inplace=True)

get_wdpa.head()

,location_id,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,shallow
0,wdpa_0,0.66,4.14,0.90,0.00,0.29,0.50,2.15,0.34,16.85,28.35,0.45,0.03,5.94,2.16,37.23
1,wdpa_1,0.02,0.42,0.04,97.94,0.00,0.85,0.00,0.06,0.48,0.19,0.00,0.00,0.00,0.00,0.00
2,wdpa_2,0.52,13.52,0.37,78.53,0.00,0.16,0.09,0.07,4.03,2.68,0.03,0.00,0.00,0.00,0.00
3,wdpa_3,0.33,9.60,0.71,0.00,83.37,0.39,0.00,0.03,1.05,1.63,2.88,0.00,0.00,0.00,0.00
4,wdpa_4,0.71,1.01,0.14,83.23,0.00,0.98,0.00,0.62,5.76,7.41,0.14,0.00,0.00,0.00,0.00


In [52]:
get_wdpa_long = get_wdpa.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_wdpa_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_wdpa_long.head(5)

,location_id,wetland_type,percentage
20240,wdpa_0,agri,0.45
12144,wdpa_0,eph_lake,2.15
18216,wdpa_0,ephinuveg,28.35
4048,wdpa_0,ephopen,0.90
8096,wdpa_0,lake,0.29


## Wetland types by hydrobasin

In [58]:
file_path = f'{gcs_path}/{analysis_data_folder}/hydro_basin_areas.csv'
get_hydrobasins = pd.read_csv(file_path)
#replace column names with the dictionary
get_hydrobasins = get_hydrobasins.rename(columns=cols_dict)

#group categories
get_hydrobasins['seas_lake'] = get_hydrobasins['seas_lake'] + get_hydrobasins['15']
get_hydrobasins['saline'] = get_hydrobasins['saline'] + get_hydrobasins['16']
get_hydrobasins['agri'] = get_hydrobasins['agri'] + get_hydrobasins['31'] + get_hydrobasins['33']
get_hydrobasins['reservoir'] = get_hydrobasins['reservoir'] + get_hydrobasins['35'] + get_hydrobasins['36']

get_hydrobasins = get_hydrobasins.drop(columns=['15', '16', '31', '33', '35', '36', 'other'])

get_hydrobasins['total'] = get_hydrobasins[wetland_types].sum(axis=1)
get_hydrobasins['basin_name'] = get_hydrobasins['Country'] + ' Basin'

get_hydrobasins = get_hydrobasins.merge(locations_basins[['name', 'id']], left_on='basin_name', right_on='name', how='right')
get_hydrobasins = get_hydrobasins.drop(columns=['name', 'basin_name'])
get_hydrobasins.rename(columns={'id': 'location_id'}, inplace=True)

#calculate percantage of each wetland type per country
for wetland in wetland_types:
    get_hydrobasins[wetland + '_perc'] = round(get_hydrobasins[wetland] / get_hydrobasins['total'] * 100, 2)

get_hydrobasins.drop(columns= wetland_types + ['total'], inplace=True)
get_hydrobasins.columns = get_hydrobasins.columns.str.replace('_perc', '')
get_hydrobasins.drop(columns=['Country'], inplace=True)

get_hydrobasins.head()

,location_id,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,shallow
0,hydrobasin_red_sea_basin,2.68,22.80,4.75,0.05,0.04,3.46,0.28,0.01,3.43,2.43,0.09,3.12,2.87,18.17,35.80
1,hydrobasin_horn_of_africa_basin,65.56,20.23,0.84,0.00,0.00,0.00,0.02,0.00,0.35,0.40,0.01,0.01,1.24,2.21,9.13
2,hydrobasin_juba_shibeli_basin,3.25,30.08,14.22,0.04,0.05,0.18,1.47,0.86,14.36,22.65,8.38,4.46,0.00,0.00,0.00
3,hydrobasin_tana_basin,2.80,9.04,5.12,0.08,1.07,0.31,0.33,0.63,11.92,22.26,17.54,3.64,15.41,5.26,4.60
4,hydrobasin_congo_basin,5.86,1.11,1.14,0.00,0.00,0.00,0.15,5.76,26.06,59.60,0.32,0.00,0.00,0.00,0.00


In [61]:
get_hydrobasins_long = get_hydrobasins.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_hydrobasins_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_hydrobasins_long.head(5)

,location_id,wetland_type,percentage
174,hydrobasin_congo_basin,agri,0.32
106,hydrobasin_congo_basin,eph_lake,0.15
157,hydrobasin_congo_basin,ephinuveg,59.60
38,hydrobasin_congo_basin,ephopen,1.14
72,hydrobasin_congo_basin,lake,0.00


## Wetland types Sahel

In [64]:
file_path = f'{gcs_path}/{analysis_data_folder}/global_areas.csv'
get_sahel = pd.read_csv(file_path)
#replace column names with the dictionary
get_sahel = get_sahel.rename(columns=cols_dict)

#group categories
get_sahel['seas_lake'] = get_sahel['seas_lake'] + get_sahel['15']
get_sahel['saline'] = get_sahel['saline'] + get_sahel['16']
get_sahel['agri'] = get_sahel['agri'] + get_sahel['31'] + get_sahel['33']
get_sahel['reservoir'] = get_sahel['reservoir'] + get_sahel['35'] + get_sahel['36']

get_sahel = get_sahel.drop(columns=['15', '16', '31', '33', '35', '36', 'other'])

get_sahel['total'] = get_sahel[wetland_types].sum(axis=1)

get_sahel['location_id'] = 'global_sahel'



#calculate percantage of each wetland type per country
for wetland in wetland_types:
    get_sahel[wetland + '_perc'] = round(get_sahel[wetland] / get_sahel['total'] * 100, 2)

get_sahel.drop(columns= wetland_types + ['total'], inplace=True)
get_sahel.columns = get_sahel.columns.str.replace('_perc', '')
get_sahel.drop(columns=['Country'], inplace=True)
get_sahel

,location_id,permopen,seasopen,ephopen,saline,lake,seas_lake,eph_lake,perinuveg,seainuveg,ephinuveg,agri,reservoir,mangrove,mudflat,shallow
0,global_sahel,2.59,8.54,2.84,2.16,3.57,1.17,0.11,0.63,31.65,26.0,13.61,3.3,1.02,0.26,2.54


In [65]:
get_sahel_long = get_sahel.melt(id_vars=['location_id'], var_name='wetland_type', value_name='percentage')
get_sahel_long.sort_values(['location_id', 'wetland_type'], inplace=True)
get_sahel_long.head(5)

,location_id,wetland_type,percentage
10,global_sahel,agri,13.61
6,global_sahel,eph_lake,0.11
9,global_sahel,ephinuveg,26.00
2,global_sahel,ephopen,2.84
4,global_sahel,lake,3.57


In [66]:
print(len(get_wetlands_long['wetland_type'].unique()))
print(len(get_wdpa_long['wetland_type'].unique()))
print(len(get_hydrobasins_long['wetland_type'].unique()))
print(len(get_sahel_long['wetland_type'].unique()))

15
15
15
15


## Combine all data

In [67]:
get_combined = pd.concat([get_wetlands_long, get_wdpa_long, get_hydrobasins_long, get_sahel_long], ignore_index=True)

In [68]:
print(len(get_combined['wetland_type'].unique()))
get_combined['wetland_type'].unique()

15


array(['agri', 'eph_lake', 'ephinuveg', 'ephopen', 'lake', 'mangrove',
       'mudflat', 'perinuveg', 'permopen', 'reservoir', 'saline',
       'seainuveg', 'seas_lake', 'seasopen', 'shallow'], dtype=object)

In [69]:
#Replace Na with 0
get_combined['percentage'] = get_combined['percentage'].replace(np.nan, 0)

In [70]:
color_dict = {'agri':"#ff9067",
              'eph_lake': "#9499ff",
              'ephinuveg': "#9ec59e",
              'ephopen': "#f6ffff",
              'lake': "#000dff",
              'mangrove': "#663399",
              'mudflat': "#a0522d",
              'perinuveg': "#006400",
              'permopen': "#7acaff",
              'reservoir': "#56b98b",
              'saline': "#cfe875",
              'seainuveg': "#8bc500",
              'seas_lake': "#4747ff",
              'seasopen': "#e2ffff",
              'shallow': "#ccffdf",}

In [71]:
indicator_data_list = []

for location in get_combined['location_id'].unique():
    df_location = get_combined[get_combined['location_id'] == location]
    df_location.reset_index(drop=True, inplace=True)
    data_list = []
    for i in range(len(df_location)):
        data_list.append({
            "id":"wetland_type_" + location + "_" + str(i),
            "label": df_location.loc[i, 'wetland_type'],
            "value": df_location.loc[i, 'percentage'],
            "type":"",
            "group": "",
            "color": color_dict.get(df_location.loc[i, 'wetland_type'], "#000000"),
            "format": "number",
            "unit": "%"
        })
    data_json = json.dumps(data_list, indent=2)
    temp_dict = {"id": "wetland-types-" + location,
                 "location": location,
                 "indicator": "wetland-types-get",
                 "data": json.loads(data_json),
                 "locale": {"en": {"labels":{
                     'agri':'Agriculture',
                     'eph_lake':'Ephemeral Lake',
                     'ephinuveg':'Ephemeral Inundated Vegetation',
                     'ephopen':'Ephemeral Open Water',
                     'lake':'Lake',
                     'mangrove':'Mangrove',
                     'mudflat':'Mudflat',
                     'perinuveg':'Permanently Inundated Vegetation',
                     'permopen':'Permanent Open Water',
                     'reservoir':'Reservoir',
                     'saline':'Saline',
                     'seainuveg':'Seasonally Inundated Vegetation',
                     'seas_lake':'Seasonal Lake',
                     'seasopen':'Seasonal Open Water',
                     'shallow':'Shallow Coast'
                    }
                    }},
                }
    indicator_data_list.append(temp_dict)


In [72]:
print(json.dumps(indicator_data_list[0], indent=2))

{
  "id": "wetland-types-adminregion_ben",
  "location": "adminregion_ben",
  "indicator": "wetland-types-get",
  "data": [
    {
      "id": "wetland_type_adminregion_ben_0",
      "label": "agri",
      "value": 11.95,
      "type": "",
      "group": "",
      "color": "#ff9067",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_1",
      "label": "eph_lake",
      "value": 0.08,
      "type": "",
      "group": "",
      "color": "#9499ff",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_2",
      "label": "ephinuveg",
      "value": 38.36,
      "type": "",
      "group": "",
      "color": "#9ec59e",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wetland_type_adminregion_ben_3",
      "label": "ephopen",
      "value": 3.18,
      "type": "",
      "group": "",
      "color": "#f6ffff",
      "format": "number",
      "unit": "%"
    },
    {
      "id": "wet

### Save to json

In [73]:
seeding_json = json.load(open('../../app-initial-data/indicator-data.json'))
#remove all entries with id that starts with "wetland-types-" and "wetland_types_"
seeding_json = [entry for entry in seeding_json if not entry['id'].startswith('wetland-types-') and not entry['id'].startswith('wetland_types_')]

# Append the new data, if id exists update it
existing_ids = {entry['id'] for entry in seeding_json}
for new_entry in indicator_data_list:
    if new_entry['id'] in existing_ids:
        seeding_json = [entry if entry['id'] != new_entry['id'] else new_entry for entry in seeding_json]
    else:
        seeding_json.append(new_entry)  
with open('../../app-initial-data/indicator-data.json', 'w') as f:
    json.dump(seeding_json, f, indent=2)